# Selección de columnas para la red bayesiana — ENMT

**Proyecto:** transporte_UNAM · **Etapa:** 02 — selección de variables para los 4 queries
**Entrada:** `data/processed/enmt_limpio.csv` + `data/processed/diccionario.json` (salidas de `01_limpieza_enmt.ipynb`)
**Salida:** `data/processed/enmt_bn.csv` (subconjunto crudo para modelar) + `data/processed/mapa_nodos.csv`

---

## Por qué este notebook

El dataset **analítico** (`enmt_analitico.csv`) no sirve para esta etapa: su recorte temático deja fuera
la victimización (`p25_*`), la ocupación (`h21_*`) y la calificación por modo (`p1c_*`), que son justo lo que
piden los queries. Partimos entonces del **completo** (`enmt_limpio.csv`).

Como los nombres de la base ya están normalizados y no siempre coinciden con el codebook, aquí **no
hardcodeamos** la lista final: para cada nodo damos columnas candidatas y el notebook reporta cuáles
existen de verdad y con qué cobertura. Tú confirmas la lista antes de modelar.

Los 4 queries y sus nodos:

1. Usuarios de metro vs. resto de TP — percepción de **seguridad** y **efectividad**.
2. **Patrones vs. profesionistas** — percepción del **costo**.
3. **Escolaridad** → **modo principal**.
4. Entre usuarios de TP — **asalto en TP** según **ingreso** ≷ promedio.


## 1. Configuración y carga

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 170)

# Raíz robusta (igual criterio que 01_limpieza)
BASE = Path.cwd()
if not (BASE / "data").is_dir():
    BASE = BASE.parent
assert (BASE / "data").is_dir(), f"No encuentro data/ desde {Path.cwd()}"

DIR_PROC = BASE / "data" / "processed"
RUTA_LIMPIO = DIR_PROC / "enmt_limpio.csv"
RUTA_DICC = DIR_PROC / "diccionario.json"

faltan = [p.name for p in (RUTA_LIMPIO, RUTA_DICC) if not p.exists()]
assert not faltan, (
    f"No existen {faltan} en {DIR_PROC}. "
    "data/processed/ está en .gitignore: corre 01_limpieza_enmt.ipynb para regenerarlos."
)

df = pd.read_csv(RUTA_LIMPIO, low_memory=False)
DICC = json.loads(RUTA_DICC.read_text(encoding="utf-8"))["variables"]

def etiqueta(col):
    return DICC.get(col, {}).get("etiqueta", "")

print(f"Base completa: {df.shape[0]:,} filas x {df.shape[1]:,} columnas")
print(f"Diccionario  : {len(DICC):,} variables")

Base completa: 1,191 filas x 556 columnas
Diccionario  : 556 variables


## 2. Mapa de nodos → columnas candidatas

Para cada nodo listamos las columnas candidatas por orden de preferencia. `exact` busca un nombre;
`patron` busca todas las columnas que casen una expresión regular (baterías como `p1a_*`).

In [2]:
# nodo, query, tipo, candidatas (en orden de preferencia), nota
MAPA = [
    # --- contexto / confusores ---
    ("region",        "conf", "exact",  ["region"], "región geográfica"),
    ("tam_loc",       "conf", "exact",  ["tam_loc"], "tamaño de localidad"),
    ("sexo",          "conf", "exact",  ["sexo", "sd1"], "sexo del informante"),
    ("edad",          "conf", "exact",  ["edad_1", "sd2"], "edad (numérica -> agrupar)"),
    ("ponderador",    "todos","exact",  ["pondi2", "pondi", "pondi_v", "pondi_h"], "factor de expansión"),
    # --- socioeconómicas ---
    ("escolaridad",   "Q3",   "exact",  ["escol", "sd4"], "driver de Q3"),
    ("ingreso_ind",   "Q4",   "exact",  ["ing_ind", "sd13"], "ingreso individual (numérico)"),
    ("ingreso_fam",   "Q4",   "exact",  ["ing_fam", "sd15"], "respaldo de ingreso"),
    ("cond_act",      "Q2",   "exact",  ["cond_act"], "trabaja/no (fallback de ocupación)"),
    ("ocupacion",     "Q2",   "patron", [r"^h21_\d+$"], "roster: ocupación por integrante"),
    # --- uso de transporte (modo principal / universo TP) ---
    ("uso_modos",     "Q1,Q3,Q4", "patron", [r"^p1a_\d+$"], "frecuencia de uso por modo"),
    # --- percepción (esquema real p17_*; alterno per-modo p1c_*) ---
    ("perc_seguridad","Q1",   "exact",  ["p17_4"], "TP seguro/inseguro"),
    ("perc_eficiente","Q1",   "exact",  ["p17_1"], "TP eficiente/ineficiente"),
    ("perc_rapido",   "Q1",   "exact",  ["p17_2"], "TP rápido/lento"),
    ("perc_costo",    "Q2",   "exact",  ["p17_3"], "TP barato/caro"),
    ("perc_seg_modo", "Q1-alt","patron",[r"^p1c_\d+_2$"], "alterno: seguridad por modo"),
    ("perc_costo_modo","Q2-alt","patron",[r"^p1c_\d+_6$"], "alterno: costo por modo"),
    # --- victimización ---
    ("asalto_tp",     "Q4",   "exact",  ["p25_1_2"], "víctima de asalto en TP"),
]
print(f"Nodos a resolver: {len(MAPA)}")

Nodos a resolver: 18


## 3. Resolución y reporte de cobertura contra la base real

In [3]:
def cobertura(col):
    s = df[col]
    return {
        "col": col,
        "pct_nulos": round(float(s.isna().mean()) * 100, 1),
        "n_unicos": int(s.nunique(dropna=True)),
        "tipo": str(s.dtype),
        "etiqueta": etiqueta(col)[:70],
    }

filas, faltantes = [], []
for nodo, query, tipo, cands, nota in MAPA:
    if tipo == "exact":
        elegida = next((c for c in cands if c in df.columns), None)
        cols = [elegida] if elegida else []
    else:  # patron
        cols = sorted(c for c in df.columns for p in cands if re.fullmatch(p, c))
    if not cols:
        faltantes.append((nodo, query, cands))
        filas.append({"nodo": nodo, "query": query, "estado": "FALTA",
                      "col": " | ".join(cands), "pct_nulos": None,
                      "n_unicos": None, "tipo": None, "etiqueta": nota})
        continue
    for c in cols:
        r = cobertura(c)
        r.update({"nodo": nodo, "query": query, "estado": "ok"})
        filas.append(r)

reporte = pd.DataFrame(filas)[
    ["nodo", "query", "estado", "col", "pct_nulos", "n_unicos", "tipo", "etiqueta"]
]
# Para baterías, resumimos (cuántas columnas y cobertura media) además del detalle
resumen_bat = (reporte[reporte["nodo"].isin(["uso_modos","ocupacion","perc_seg_modo","perc_costo_modo"])]
               .groupby("nodo")
               .agg(n_cols=("col","size"), pct_nulos_medio=("pct_nulos","mean"))
               .round(1))

print("── Nodos de un solo ítem ──")
display(reporte[~reporte["nodo"].isin(resumen_bat.index)].reset_index(drop=True))
print("\n── Baterías (resumen) ──")
display(resumen_bat)

if faltantes:
    print("\n⚠ NODOS NO ENCONTRADOS EN LA BASE (revisar nombre o si están en el crudo):")
    for nodo, query, cands in faltantes:
        print(f"   · {nodo} [{query}] — buscaba: {cands}")

── Nodos de un solo ítem ──


,nodo,query,estado,col,pct_nulos,n_unicos,tipo,etiqueta
0,region,conf,ok,region,0.0,4,int64,Región
1,tam_loc,conf,ok,tam_loc,0.0,4,int64,Tamaño de localidad
2,sexo,conf,ok,sexo,0.0,2,int64,Sexo
3,edad,conf,ok,edad_1,0.0,6,int64,Edad
4,ponderador,todos,ok,pondi2,0.0,933,int64,Factor de expansión individual
5,escolaridad,Q3,ok,escol,0.1,5,float64,Escolaridad
6,ingreso_ind,Q4,ok,ing_ind,0.1,5,float64,Ingreso Individual
7,ingreso_fam,Q4,ok,ing_fam,0.0,8,int64,Ingreso familiar
8,cond_act,Q2,ok,cond_act,0.6,2,float64,Condición de actividad
9,perc_seguridad,Q1,ok,p17_4,0.0,2,int64,"17 Usted, ¿cómo considera el transporte públic..."



── Baterías (resumen) ──


,n_cols,pct_nulos_medio
nodo,,
ocupacion,6,76.6
perc_costo_modo,11,90.7
perc_seg_modo,21,93.5
uso_modos,22,0.8


## 4. Puntos abiertos que el reporte de arriba te va a confirmar

- **`asalto_tp` (`p25_1_2`)** y **`ocupacion` (`h21_*`)**: si salen como `FALTA`, es que no llegaron al
  `enmt_limpio.csv` (pudieron quedar 100 % nulos o no venir en el crudo). En ese caso hay que revisar el
  CSV original o replantear el query.
- **Enlace ocupación → individuo (Q2)**: `h21_*` es del *roster* (una por integrante). Necesitas saber qué
  renglón es el informante. `h8_*` (que traía los nombres) se borró por PII, así que busca en el crudo una
  variable de "número de renglón del seleccionado". Si no existe, Q2 se queda con `cond_act` (trabaja/no),
  que **no** distingue patrón de profesionista — habría que ajustar el query.
- **Decisión 2 (percepción del modo principal)**: con `p17_*` la percepción es del **TP en general**, no de
  tu modo. Si quieres conservar "percepción del modo principal", tienes que usar la batería `p1c_*`
  (`perc_seg_modo`, `perc_costo_modo`) — revisa su cobertura antes de decidir; suele tener muchos nulos por
  salto de pregunta.
- **`escol`**: aparece sin etiquetas de valor; el reporte te dice si es nivel codificado o años. Defínelo
  antes de agrupar para Q3.

## 5. Derivar el modo principal (preview)

`modo_principal` no es una columna: se deriva de la batería `p1a_*` (1=cotidianamente, 2=ocasionalmente,
3=nunca; NS/NC ya son NaN). Tomamos, por persona, el modo con la frecuencia más alta (código más bajo) y lo
agrupamos. Las etiquetas de modo se leen del propio diccionario para no hardcodear el orden.

In [4]:
p1a_cols = sorted(c for c in df.columns if re.fullmatch(r"p1a_\d+", c))

def grupo_modo(lbl):
    l = lbl.lower()
    if re.search(r"metro|tren urbano|tren ligero|suburbano", l): return "metro"
    if re.search(r"\btren\b|brt|metrob|eléctric|electric|trolebus|tranv", l): return "tp_masivo"
    if re.search(r"camión|camion|microb|colectivo|combi|autobús|autobus", l): return "tp_concesionado"
    if re.search(r"taxi|bicitaxi|mototaxi", l): return "taxi"
    if re.search(r"automóvil|automovil|moto", l): return "auto_moto"
    if re.search(r"bicicleta|patín|patin|caminar|pie", l): return "activo"
    return "otro"

MODO_GRUPO = {c: grupo_modo(etiqueta(c)) for c in p1a_cols}
print("Modos detectados (columna -> grupo):")
for c in p1a_cols:
    print(f"  {c:9} {grupo_modo(etiqueta(c)):16} {etiqueta(c)[:60]}")

# Matriz de frecuencias; el modo principal es el de código mínimo por fila
frec = df[p1a_cols]
idx_min = frec.idxmin(axis=1)                  # columna con la frecuencia más alta
freq_min = frec.min(axis=1)
modo_principal = idx_min.map(MODO_GRUPO)
modo_principal[freq_min.isna() | (freq_min >= 3)] = np.nan  # nadie con uso cotidiano/ocasional

df["modo_principal"] = modo_principal
df["metro_principal"] = np.where(modo_principal.isna(), np.nan,
                                 (modo_principal == "metro").astype("Int64"))
tp = {"metro", "tp_masivo", "tp_concesionado"}
df["usuario_tp"] = modo_principal.isin(tp).astype("Int64")

print("\nDistribución de modo_principal:")
display(df["modo_principal"].value_counts(dropna=False).to_frame("n"))

Modos detectados (columna -> grupo):
  p1a_1     tp_masivo        1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_10    otro             1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_11    otro             1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_12    auto_moto        1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_13    otro             1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_14    otro             1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_15    auto_moto        1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_16    activo           1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_17    activo           1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_18    otro             1a  ¿Con que frecuencia utilizas los siguientes medios de tr
  p1a_19    otro             1a  ¿Con que frecuencia utilizas l

,n
modo_principal,
tp_concesionado,579
auto_moto,320
activo,77
taxi,67
metro,65
otro,62
tp_masivo,15
NaN,6


## 6. Construir y exportar el subconjunto para la red

In [5]:
# Columnas crudas confirmadas (las presentes) + derivadas
cols_ok = [r["col"] for _, r in reporte.iterrows() if r["estado"] == "ok"]
derivadas = ["modo_principal", "metro_principal", "usuario_tp"]
cols_bn = list(dict.fromkeys(cols_ok + derivadas))   # sin duplicados, en orden

df_bn = df[cols_bn].copy()

RUTA_BN = DIR_PROC / "enmt_bn.csv"
RUTA_MAPA = DIR_PROC / "mapa_nodos.csv"
df_bn.to_csv(RUTA_BN, index=False, encoding="utf-8")
reporte.to_csv(RUTA_MAPA, index=False, encoding="utf-8")

print(f"✓ {RUTA_BN.name}: {df_bn.shape[0]:,} x {df_bn.shape[1]:,}")
print(f"✓ {RUTA_MAPA.name}: mapa nodo→columna con cobertura")
print(f"\nColumnas incluidas ({len(cols_bn)}):\n  {cols_bn}")

✓ enmt_bn.csv: 1,191 x 77


✓ mapa_nodos.csv: mapa nodo→columna con cobertura

Columnas incluidas (77):
  ['region', 'tam_loc', 'sexo', 'edad_1', 'pondi2', 'escol', 'ing_ind', 'ing_fam', 'cond_act', 'h21_1', 'h21_2', 'h21_3', 'h21_4', 'h21_5', 'h21_6', 'p1a_1', 'p1a_10', 'p1a_11', 'p1a_12', 'p1a_13', 'p1a_14', 'p1a_15', 'p1a_16', 'p1a_17', 'p1a_18', 'p1a_19', 'p1a_2', 'p1a_20', 'p1a_21', 'p1a_22', 'p1a_3', 'p1a_4', 'p1a_5', 'p1a_6', 'p1a_7', 'p1a_8', 'p1a_9', 'p17_4', 'p17_1', 'p17_2', 'p17_3', 'p1c_10_2', 'p1c_11_2', 'p1c_12_2', 'p1c_13_2', 'p1c_14_2', 'p1c_15_2', 'p1c_16_2', 'p1c_17_2', 'p1c_18_2', 'p1c_19_2', 'p1c_1_2', 'p1c_21_2', 'p1c_22_2', 'p1c_2_2', 'p1c_3_2', 'p1c_4_2', 'p1c_5_2', 'p1c_6_2', 'p1c_7_2', 'p1c_8_2', 'p1c_9_2', 'p1c_10_6', 'p1c_11_6', 'p1c_1_6', 'p1c_2_6', 'p1c_3_6', 'p1c_4_6', 'p1c_5_6', 'p1c_6_6', 'p1c_7_6', 'p1c_8_6', 'p1c_9_6', 'p25_1_2', 'modo_principal', 'metro_principal', 'usuario_tp']


## 7. Siguientes pasos (notebook 03)

1. Cerrar los puntos abiertos de la sección 4 (ocupación/enlace, decisión 2, tipo de `escol`).
2. **Recodificar** cada nodo a las categorías del modelo:
   - `escolaridad` → básica / media superior / superior
   - `ingreso_bin` → ≷ promedio **entre usuarios de TP**
   - percepción binaria de `p17_*` ya viene lista; si usas `p1c_*` o escalas 0–10, colapsa a 3 niveles
   - `ocupacion` → patrón / profesionista / otro
3. Filtrar universos por query (usuarios de TP para Q1 y Q4).
4. Construir las 3 DAGs con `pgmpy` sobre los nodos ya recodificados.
